# Fraud modelling with XGBoost and SHAP

This notebook compares Logistic Regression with XGBoost on the chronological test period, then uses SHAP to explain the fitted XGBoost model.

> **Research boundary:** this experiment uses synthetic claims and synthetic fraud labels. It demonstrates the method and software pipeline rather than production performance.

## 1. Set up the project

The notebook imports the tested project functions instead of maintaining a second copy of the training logic.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
REPOSITORY_URL = "https://github.com/ganesh1997oli/Decentralized-Claims-Registry"

if IN_COLAB:
    PROJECT_ROOT = Path("/content/Decentralized-Claims-Registry")
    if not PROJECT_ROOT.exists():
        subprocess.run(["git", "clone", REPOSITORY_URL, str(PROJECT_ROOT)], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_ROOT / "model/requirements.txt")],
        check=True,
    )
    os.chdir(PROJECT_ROOT)
else:
    current = Path.cwd().resolve()
    PROJECT_ROOT = next(
        (path for path in (current, *current.parents) if (path / "model/research_pipeline.py").exists()),
        None,
    )
    if PROJECT_ROOT is None:
        raise RuntimeError("Start Jupyter from inside the repository.")
    os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project: {PROJECT_ROOT}")
print(f"Running in Colab: {IN_COLAB}")

## 2. Load the pinned dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay
from IPython.display import Image, display

from model.download_dataset import DEFAULT_DATASET_PATH, download_dataset
from model.research_pipeline import (
    create_shap_summary,
    load_claims,
    train_research_models,
)

if not DEFAULT_DATASET_PATH.exists():
    download_dataset(DEFAULT_DATASET_PATH)

claims = load_claims(DEFAULT_DATASET_PATH)
print(f"Prepared rows: {len(claims):,}")
print(f"Overall synthetic fraud rate: {claims['fraud_flag'].mean():.2%}")

## 3. Train the baseline and XGBoost

Both models use the same preprocessing and chronological 70/15/15 split. Thresholds are selected on validation F1; the test period is used only for the final comparison.

In [ ]:
run = train_research_models(claims)
run.report["split"]

## 4. Compare test metrics

PR-AUC is especially useful here because fraud is the minority class. Lower Brier score means better probability calibration.

In [ ]:
model_reports = {
    "Logistic Regression": run.report["baseline_logistic_regression"],
    "XGBoost": run.report["xgboost"],
}
metric_names = ["precision", "recall", "f1", "roc_auc", "pr_auc", "brier_score", "threshold"]
metrics = pd.DataFrame(
    {name: {metric: report[metric] for metric in metric_names} for name, report in model_reports.items()}
).T
metrics.style.format("{:.3f}")

In [ ]:
comparison_metrics = ["precision", "recall", "f1", "roc_auc", "pr_auc"]
ax = metrics[comparison_metrics].T.plot(kind="bar", figsize=(10, 5), rot=0)
ax.set_ylim(0, 1)
ax.set_ylabel("Score")
ax.set_title("Performance on the untouched synthetic test period")
plt.tight_layout()
plt.show()

## 5. Inspect confusion matrices

The matrices make the operational trade-off visible: catching more fraudulent claims generally sends more legitimate claims for review.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for axis, (name, report) in zip(axes, model_reports.items()):
    ConfusionMatrixDisplay(
        confusion_matrix=np.asarray(report["confusion_matrix"]),
        display_labels=["Not fraud", "Fraud"],
    ).plot(ax=axis, colorbar=False, cmap="Blues")
    axis.set_title(name)
plt.tight_layout()
plt.show()

## 6. Explain XGBoost with SHAP

The summary plot shows which transformed features most strongly move this model's predictions. It explains model behaviour; it does not prove causation or confirm fraud.

In [ ]:
shap_path = Path("model/artifacts/xgboost-notebook/shap-summary.png")
top_features = create_shap_summary(
    run.xgboost,
    run.split.test,
    shap_path,
    sample_size=500,
)

display(pd.DataFrame(top_features))
display(Image(filename=str(shap_path)))

## 7. Optional: save reproducible artifacts

Change `SAVE_ARTIFACTS` to `True` only when you want the fitted pipeline, metadata and SHAP chart written under `model/artifacts/`.

In [ ]:
from model.download_dataset import DATASET_REPOSITORY, DATASET_REVISION, file_sha256
from model.research_pipeline import save_training_run

SAVE_ARTIFACTS = False
if SAVE_ARTIFACTS:
    output_dir = Path("model/artifacts/xgboost-african-motor-v1")
    metadata = save_training_run(
        run,
        output_dir,
        dataset_reference=f"{DATASET_REPOSITORY}@{DATASET_REVISION}",
        dataset_sha256=file_sha256(DEFAULT_DATASET_PATH),
    )
    print(f"Saved artifacts to {output_dir}")
else:
    print("Artifact export skipped.")

## Interpretation checklist

- Compare XGBoost with the simpler baseline instead of assuming it is better.
- Discuss precision and recall as a review-workload trade-off.
- Report that thresholds were chosen using validation data only.
- Treat SHAP values as explanations of the fitted model, not evidence of causality.
- State clearly that real, independently labelled claims are required before deployment.